# خط معالجة صوتية إنتاجي متكامل — من التنزيل إلى بيانات جاهزة لتدريب RVC v2

**الهدف:** تحويل أرشيف رسائل صوتية خام (Voice.zip) إلى مجموعة بيانات نظيفة عالية الجودة
جاهزة مباشرة لتدريب نموذج استنساخ الصوت **RVC v2**، بأقوى سلسلة أدوات متاحة لكل مرحلة.

**خريطة الخط (10 مراحل):**

| # | المرحلة | الأداة/النموذج (الأفضل في فئته) |
|---|---------|--------------------------------|
| 0 | تنزيل قوي + تحقق CRC | `requests` streaming + `zipfile` |
| 1 | جرد وفك ترميز موحد | `librosa`/`soundfile` (+ ffmpeg احتياطي) |
| 2 | إزالة التكرارات | بصمة md5 + بصمة طيفية بالارتباط المتقاطع |
| 3 | تشخيص كامل قبل المعالجة | VAD Silero v5 + `pyloudnorm` EBU R128 |
| 4 | عزل الصوت البشري (شرطي) | BS-RoFormer ثم MelBand RoFormer ثم MDX-Net (عبر `audio-separator`) |
| 5 | إزالة الضوضاء | `UVR-DeNoise` (مع بديل `noisereduce`) |
| 6 | إزالة الصدى والارتداد | `UVR-DeEcho-DeReverb` |
| 7 | قص ذكي + تقطيع 3–12 ث | Silero VAD v5 |
| 8 | توحيد الجهارة | `pyloudnorm` (LUFS، معيار EBU R128) |
| 9 | إعادة عينة + تصدير نهائي | `soxr` VHQ → WAV أحادي 48kHz / 16-bit |

**الاستخدام:** شغّل الخلايا بالترتيب من الأعلى إلى الأسفل. عدّل خلية `CONFIG` قبل أي شيء —
كل قيمة قابلة للضبط موجودة هناك فقط. المسارات كلها نسبية (يعمل في Jupyter / JupyterLab /
VS Code / Colab / Binder دون تعديل). الخلايا آمنة لإعادة التشغيل: التنزيل يُتخطى إن وُجد،
وكل مرحلة ثقيلة تحفظ مخرجاتها في ذاكرة مؤقتة وتتجاوز ما تم إنجازه سابقًا.

**متطلبات:** لا يفترض الدفتر شيئًا عن بيئتك (لا GPU، لا حزم مثبتة مسبقًا): خلية الفحص تكشف
القدرات، وخلية الاعتماديات تثبّت الناقص اختياريًا. على GPU تكون مراحل الفصل أسرع بعشرات
المرات، وعلى CPU تعمل كاملة لكن ببطء.

## 1 — الإعداد

### 1.1 التكوين المركزي `CONFIG`

المكان الوحيد لتغيير السلوك. لاحظ:

- `zip_url` رابط Google Drive الموقّت — قد ينتهي؛ عندئذٍ نزّل الملف يدويًا وضعه بجوار
  الدفتر باسم `Voice.zip` وسيُلتقط تلقائيًا من `manual_zip_names`.
- `separation_mode`: `"auto"` = فحص استباقي حقيقي لكل ملف (نموذج فصل خفيف على عينة 20
  ثانية، وقياس طاقة «الآلات» المستخرجة) ثم فصل كامل للملفات التي ثبت وجود موسيقى فيها
  فقط؛ `"all"` = فصل الجميع؛ `"none"` = تعطيل؛ أو قائمة معرّفات مثل `["a001"]`.
- `quick_test`: وضع التحقق السريع (ملفات قليلة قصيرة) لتجربة الخط كاملًا قبل تشغيل
  كامل — لا يُستخدم في التشغيل النهائي.

In [ ]:
from pathlib import Path

CONFIG = {
    # ---------------- المصدر ----------------
    # رابط Google Drive الموقّت (سطر واحد كي يبقى قابلًا لحقن papermill)
    "zip_url": "https://drive.usercontent.google.com/download?id=1dXino3_rN6uV6RUGJpyfo3U4_bxxBXs8&export=download&authuser=0&confirm=t",
    "manual_zip_names": ["Voice.zip"],   # أسماء يقبلها الدفتر إن نُزّل الملف يدويًا
    "expected_zip_mb": 30,               # حجم أدنى معقول لملف ZIP (فحص سلامة)

    # ---------------- المسارات (نسبية فقط) ----------------
    "raw_dir": "data/raw",               # الصوت الخام المستخرج
    "work_dir": "work",                  # المخرجات الوسيطة والذاكرة المؤقتة
    "out_dir": "output/rvc_dataset",     # مجموعة البيانات النهائية
    "models_dir": "models_cache",        # ذاكرة نماذج UVR المؤقتة

    # ---------------- الهدف النهائي (معايير RVC v2) ----------------
    "target_sr": 48000,                  # RVC v2 يدعم حتى 48k — الأعلى جودة
    "bit_depth": 16,                     # PCM 16-bit
    "target_lufs": -20.0,                # جهارة موحدة (LUFS مدمجة، EBU R128)
    "peak_ceiling_db": -1.0,             # سقف القمة الآمن (dBFS)

    # ---------------- VAD والتقطيع ----------------
    "vad_threshold": 0.5,                # عتبة Silero VAD
    "vad_min_speech_ms": 250,            # أقصر كلام يُعتد به
    "vad_min_gap_ms": 350,               # فجوات أقصر من هذا تُدمج داخل المقطع
    "edge_pad_s": 0.15,                  # هامش أمان حول كل مقطع كلام
    "merge_gap_s": 1.2,                  # دمج مقاطع متجاورة تفصلها فجوة <= هذه القيمة
    "min_clip_s": 3.0,                   # أقصر مقطع تدريب مطلوب
    "max_clip_s": 12.0,                  # أطول مقطع تدريب مسموح
    "drop_below_s": 1.5,                 # ما دون هذا يُهمَل

    # ---------------- مراحل المعالجة ----------------
    "separation_mode": "auto",           # auto | all | none | قائمة معرفات
    "probe_model": "4_HP-Vocal-UVR.pth", # نموذج الفحص الاستباقي (خفيف وسريع)
    "probe_seconds": 20,                 # طول عينة الفحص من منتصف الملف
    "probe_flag_db": -25.0,              # نسبة (آلات/غناء) فوقها يُعلن وجود موسيقى
    "separation_models": [               # سلسلة النماذج بالترتيب (أفضلها أولًا)
        "model_bs_roformer_ep_368_sdr_12.9628.ckpt",   # BS-RoFormer — SDR الأعلى
        "model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt",  # MelBand RoFormer
        "UVR-MDX-NET-Inst_HQ_3.onnx",   # MDX-Net — الاحتياط الأخف
    ],
    "denoise_engine": "auto",            # auto | uvr | noisereduce | none
    "denoise_model": "UVR-DeNoise.pth",  # معيار مجتمع RVC لإزالة الضوضاء
    "noisereduce_prop_decrease": 0.75,   # شدة البديل الضعيف (احترازية)
    "dereverb_enable": True,
    "dereverb_model": "UVR-DeEcho-DeReverb.pth",

    # ---------------- سلوك التنفيذ ----------------
    "auto_install": True,               # تثبيت الحزم الناقصة تلقائيًا
    "use_cache": True,                  # تخطي المراحل المنجزة (إعادة تشغيل آمنة)
    "package_zip": True,                # ضغط المخرجات النهائية في ZIP واحد
    "seed": 42,
    "quick_test": False,                # True = تشغيل تحقق سريع على عينة صغيرة
    "quick_test_files": 3,
    "quick_test_seconds": 45,
}

### 1.1.1 البذرة والمسارات

**قابلية إعادة الإنتاج:** تثبيت بذرة العشوائية لـ random وnumpy (وtorch لاحقًا في خلية
الاستيرادات عند توفره)، ثم إنشاء شجرة المسارات النسبية وختْم زمن التشغيل الذي يُوثَّق
لاحقًا في بيان البيان (`manifest.json`).

In [ ]:
import random
from datetime import datetime
from pathlib import Path

random.seed(CONFIG["seed"])
try:
    import numpy as _np_early
    _np_early.random.seed(CONFIG["seed"])
except ImportError:
    pass

PATHS = {
    "raw": Path(CONFIG["raw_dir"]),
    "work": Path(CONFIG["work_dir"]),
    "out": Path(CONFIG["out_dir"]),
    "out_full": Path(CONFIG["out_dir"]) / "full_clean",
    "out_clips": Path(CONFIG["out_dir"]) / "clips",
    "out_qa": Path(CONFIG["out_dir"]) / "qa",
    "models": Path(CONFIG["models_dir"]),
    "reports": Path(CONFIG["work_dir"]) / "reports",
}
list(map(lambda k: PATHS[k].mkdir(parents=True, exist_ok=True),
         ("raw", "work", "models", "out_full", "out_clips", "out_qa", "reports")))

RUN_STAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("بدء التشغيل:", RUN_STAMP)
print("مجلد العمل:", PATHS["work"].resolve())

### 1.2 فحص البيئة

**الهدف:** كشف القدرات الفعلية بدل افتراضها (لا نستنتج من اسم البيئة شيئًا).
**المخرجات:** نظام التشغيل، المعالج، الذاكرة، توفر GPU (إن وُجد torch)، توفر ffmpeg.
**الفشل الآمن:** أي مكتبة غير مثبتة تظهر «غير معروف» دون انهيار الخلية.

In [ ]:
import os
import platform
import shutil
import sys

ENV = {
    "python": sys.version.split()[0],
    "os": f"{platform.system()} {platform.release()}",
    "machine": platform.machine(),
    "cpu_cores": os.cpu_count(),
}
try:
    import psutil
    mem = psutil.virtual_memory()
    ENV["ram_gb"] = round(mem.total / 1e9, 1)
    ENV["ram_free_gb"] = round(mem.available / 1e9, 1)
except ImportError:
    ENV["ram_gb"] = "غير معروف (psutil غير مثبت)"
try:
    import torch
    ENV["gpu"] = (torch.cuda.get_device_name(0)
                  if torch.cuda.is_available() else "لا يوجد (CPU)")
except ImportError:
    ENV["gpu"] = "غير معروف (torch غير مثبت)"
ENV["ffmpeg"] = shutil.which("ffmpeg") or "غير موجود"
ENV["frontend_hints"] = {k: v for k, v in {
    "colab": os.environ.get("COLAB_GPU"),
    "kaggle": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    "vscode": os.environ.get("VSCODE_PID"),
    "jupyterhub": os.environ.get("JUPYTERHUB_API_TOKEN") is not None,
}.items() if v}

print("\n".join(f"  {k:<14} {v}" for k, v in ENV.items()))

### 1.3 الاعتماديات

**الهدف:** ضمان وجود الحزم، مع تثبيت اختياري آمن للحاق الناقص. **الطبقتان:**

1. **أساسية** (لا يعمل الخط بدونها): numpy, soundfile, librosa, pandas, matplotlib,
   requests, tqdm, pyloudnorm, soxr.
2. **للمراحل المتقدمة** (يتحقق الخط ويتجاوزها بأمان إن غابت): `audio-separator`
   (الفصل/التنقية بنماذج UVR)، `silero-vad` (قص الكلام)، `noisereduce` (بديل إزالة
   الضوضاء)، `torch` (مطلوب لـ silero والفصل بـ RoFormer).

**ملاحظة:** `audio-separator[cpu]` يكفي على المعالج؛ على GPU ثبّت `audio-separator[gpu]`
خارج الدفتر للحصول على تسريع CUDA (الخلية تكشف الحالة ولا تفشل).

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ["numpy", "soundfile", "librosa", "pandas", "matplotlib",
            "requests", "tqdm", "pyloudnorm", "soxr"]
OPTIONAL = {
    "audio-separator": "المراحل 4-6 (فصل/تنقية UVR)",
    "silero-vad": "المرحلة 7 (قص الكلام بدقة عالية)",
    "noisereduce": "بديل إزالة الضوضاء",
    "torch": "مطلوب لـ silero-vad و RoFormer",
}

def have(pkg):
    return importlib.util.find_spec(pkg.replace("-", "_")) is not None

def pip_install(pkgs):
    print("  تثبيت:", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
                   check=False)

missing_required = [p for p in REQUIRED if not have(p)]
missing_optional = [p for p in OPTIONAL if not have(p)]
if missing_required and CONFIG["auto_install"]:
    pip_install(missing_required)
if missing_optional and CONFIG["auto_install"]:
    pip_install(missing_optional)   # silero-vad يجلب torch تلقائيًا عند غيابه

STATUS = {p: have(p) for p in REQUIRED + list(OPTIONAL)}
print("\n".join(
    f"  {'مثبت' if ok else 'غير متوفر':<10} {p}"
    + ("" if ok else "  <- " + OPTIONAL.get(p, "أساسية"))
    for p, ok in STATUS.items()))

CRITICAL_MISSING = [p for p in REQUIRED if not have(p)]
if CRITICAL_MISSING:
    raise RuntimeError(
        "حزم أساسية ناقصة: " + ", ".join(CRITICAL_MISSING) +
        " — ثبّتها يدويًا أو أعد تشغيل هذه الخلية بعد ضبط auto_install=True")

### 1.4 الاستيرادات والتسجيل

**الهدف:** استيراد مركزي محمي — لا ينكسر الدفتر لغياب مكتبة اختيارية؛ كل اختيار
مُسجَّل في أعلام `HAS_*` تحدد سلوك المراحل اللاحقة (تساقط آمن مع تحذير صريح لا صمت
ولا تظاهر بمعالجة لم تحدث). التسجيل بمُعالِج واحد يمنع تكرار السجلات عند إعادة
التشغيل، ويكتب نسخة دائمة في `work/pipeline.log`.

In [ ]:
import hashlib
import json
import logging
import shutil
import time
import warnings
from pathlib import Path

import librosa
import pandas as pd
import requests
import soundfile as sf
import soxr
import pyloudnorm as pyln
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
try:
    from silero_vad import load_silero_vad, get_speech_timestamps
    import torch as _torch
    _torch.manual_seed(CONFIG["seed"])
    HAS_SILERO = True
except ImportError:
    HAS_SILERO = False
try:
    from audio_separator.separator import Separator
    HAS_ASEP = True
except Exception:
    HAS_ASEP = False
try:
    import noisereduce as nr
    HAS_NR = True
except ImportError:
    HAS_NR = False

LOGGER = logging.getLogger("rvc_pipeline")
if not LOGGER.handlers:
    _h1 = logging.StreamHandler()
    _h2 = logging.FileHandler(PATHS["work"] / "pipeline.log", encoding="utf-8")
    _h1.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    _h2.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    LOGGER.addHandler(_h1)
    LOGGER.addHandler(_h2)
    LOGGER.setLevel(logging.INFO)

print("الاستيرادات جاهزة.")
print("  silero-vad:", "متاح" if HAS_SILERO else "غير متوفر — سيُستخدم قص طاقي أضعف")
print("  audio-separator:", "متاح" if HAS_ASEP else "غير متوفر — مراحل الفصل ستُتخطى بأمان")
print("  noisereduce:", "متاح" if HAS_NR else "غير متوفر")

### 1.5 الأدوات المساعدة المشتركة

دوال رياضية وكتابة تخدم كل المراحل: تحويلات دسيبل، **كتابة صوت ذرّية** (ملف مؤقت ثم
`os.replace` — لا يبقى ملف نصف مكتوب يُحسب نتيجة صحيحة)، بصمات SHA-256 (مع فحص وجود)،
و**حذف آمن** لمجلدات مساحة العمل فقط (يرفض المسارات خارجها).

In [ ]:
import numpy as np

def db(x):
    """تحويل خطي إلى دسيبل مع حماية من الصفر."""
    return 20.0 * np.log10(max(float(x), 1e-12))

def rms(y):
    return float(np.sqrt(np.mean(np.square(y)))) if len(y) else 0.0

def rms_db(y):
    return db(rms(y))

def atomic_write_audio(path, y, sr, subtype="FLOAT"):
    """كتابة صوت ذرّية: ملف مؤقت ثم استبدال — لا ملفات نصف مكتوبة أبدًا."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    sf.write(tmp, y, sr, subtype=subtype, format="WAV")
    tmp.replace(path)
    return path

def sha256_of_file(path, chunk=1 << 20):
    """بصمة SHA-256 مع فحص وجود واضح قبل الفتح."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"الملف غير موجود للبصمة: {path}")
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def safe_rmtree(path):
    """حذف مجلد داخل مساحة العمل فقط — يرفض رفضًا قاطعًا أي مسار خارجها."""
    p = Path(path).resolve()
    root = PATHS["work"].resolve()
    if root == p or root not in p.parents:
        raise RuntimeError(f"رفض الحذف خارج مساحة العمل: {p}")
    shutil.rmtree(p, ignore_errors=True)

AUDIO_EXT = {".ogg", ".oga", ".mp3", ".wav", ".m4a", ".mp4", ".aac",
             ".flac", ".opus", ".webm", ".wma", ".aif", ".aiff"}
QUALITY_RANK = {".wav": 0, ".flac": 1, ".ogg": 2, ".oga": 2, ".m4a": 3, ".mp4": 3,
                ".aac": 3, ".opus": 2, ".webm": 4, ".wma": 4, ".mp3": 5,
                ".aif": 0, ".aiff": 0}

RUN_META = {"run_stamp": RUN_STAMP, "config": dict(CONFIG), "env": ENV,
            "available": {"silero_vad": HAS_SILERO, "audio_separator": HAS_ASEP,
                          "noisereduce": HAS_NR, "matplotlib": HAS_MPL}}
LOGGER.info("الأدوات جاهزة. silero=%s audio-separator=%s noisereduce=%s",
            HAS_SILERO, HAS_ASEP, HAS_NR)
print("الأدوات المساعدة جاهزة (كتابة ذرّية، بصمات، حذف آمن، تصنيف جودة الصيغ).")

## 2 — المرحلة 0: التنزيل القوي

**الهدف:** جلب `Voice.zip` بثمانية ضوابط: تدفق مقطّع (لا يستهلك الذاكرة)، مهلة زمنية،
إعادة محاولة بتراجع أُسّي، كتابة ذرّية عبر `.part`، فحص حجم أدنى، والتحقق أن الرد ملف
ثنائي فعلاً وليس صفحة HTML من Google Drive (علامة انتهاء صلاحية الرابط).

**نمط الفشل المُعالَج:** انقطاع الشبكة أثناء التحميل، رابط منتهٍ، ملف مبتور. عند انتهاء
الرابط ترفع الخلية خطأً واضح الخطوات: نزّل يدويًا وضع `Voice.zip` بجوار الدفتر ثم أعد
التشغيل — الالتقاط يتم في الخلية التالية.

In [ ]:
def robust_download(url, dest, expected_mb=0, max_retries=3, chunk=1 << 20):
    """تنزيل تدفقي بمحاولات متكررة وكتابة ذرّية. يرفع خطأً واضحًا عند فشل الرد."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print("  موجود مسبقًا (ذاكرة مؤقتة):", dest.name)
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as r:
                ctype = r.headers.get("content-type", "")
                if r.status_code != 200 or ctype.startswith("text/html"):
                    raise RuntimeError(
                        "الرابط لم يعد يرجع الملف (نهاية صلاحية رابط Google Drive "
                        f"أو حد تحميل). status={r.status_code} type={ctype!r}\n"
                        "  الحل: نزّل Voice.zip يدويًا من المتصفح، ضعه بجوار الدفتر "
                        "أو في data/ ، ثم أعد تشغيل الخلية.")
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                if expected_mb and total and total / 1e6 < expected_mb:
                    raise RuntimeError(f"حجم مريب: {total/1e6:.1f}MB < {expected_mb}MB")
                with open(tmp, "wb") as f, tqdm(total=total or None, unit="B",
                                                unit_scale=True,
                                                desc=dest.name) as bar:
                    for data in r.iter_content(chunk_size=chunk):
                        f.write(data)
                        bar.update(len(data))
            if tmp.stat().st_size < 1e6:
                raise RuntimeError("الملف المُنزّل أصغر من 1MB — مبتور على الأغلب")
            tmp.replace(dest)
            print(f"  اكتمل: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return dest
        except Exception as e:
            last_err = e
            tmp.unlink(missing_ok=True)
            if attempt < max_retries and "صلاحية" not in str(e):
                wait = 2 ** attempt
                print(f"  محاولة {attempt} فشلت ({e}) — إعادة المحاولة بعد {wait}ث")
                time.sleep(wait)
            else:
                break
    raise RuntimeError(f"فشل التنزيل بعد {max_retries} محاولات: {last_err}")

### 2.1 جلب الأرشيف وفحصه واستخراجه

**الترتيب:** (1) البحث عن نسخة محلية يدوية بأسماء `manual_zip_names` بجوار الدفتر وفي
`data/` — إن وُجدت نتجاوز التنزيل كليًا. (2) تحقق `zipfile.is_zipfile` + فحص CRC لكل
عضو عبر `testzip()`. (3) استخراج الملفات **الجديدة فقط** (ما استُخرج من تشغيل سابق
يُتخطى — لا ازدواج أبدًا) بتجاهل `__MACOSX` والخردة.

In [ ]:
import zipfile

def find_manual_zip():
    """البحث عن نسخة محلية من الأرشيف قبل محاولة التنزيل."""
    names = CONFIG["manual_zip_names"]
    roots = [Path("."), Path("data"), Path.cwd()]
    for root in roots:
        for name in names:
            cand = root / name
            if cand.exists() and cand.stat().st_size > 1e6:
                return cand
    return None

ZIP_PATH = PATHS["work"] / "_download" / "Voice.zip"
local_zip = find_manual_zip()
if local_zip is not None:
    ZIP_PATH = local_zip
    print("وُجدت نسخة محلية:", local_zip)
elif ZIP_PATH.exists():
    print("موجود مسبقًا:", ZIP_PATH)
else:
    ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    ZIP_PATH = robust_download(CONFIG["zip_url"], ZIP_PATH,
                               expected_mb=CONFIG["expected_zip_mb"])

assert zipfile.is_zipfile(ZIP_PATH), "الملف ليس أرشيف ZIP صالحًا"
with zipfile.ZipFile(ZIP_PATH) as zf:
    bad = zf.testzip()          # فحص CRC لكل عضو
    if bad is not None:
        raise RuntimeError(f"عضو تالف في الأرشيف (فشل CRC): {bad}")
    members = [m for m in zf.namelist()
               if not m.endswith("/") and "__MACOSX" not in m
               and Path(m).suffix.lower() in AUDIO_EXT]
    print(f"فحص CRC سليم. أعضاء صوتية: {len(members)}")

    existing = {p.name for p in PATHS["raw"].iterdir()
                if p.suffix.lower() in AUDIO_EXT and p.stat().st_size > 1024}
    n_copied, n_skipped = 0, 0
    for m in tqdm(members, desc="استخراج", unit="ملف"):
        name = Path(m).name
        if name in existing:
            n_skipped += 1      # مستخرج من تشغيل سابق — لا تكرار
            continue
        target = PATHS["raw"] / name
        stem, suffix = target.stem, target.suffix
        k = 2
        while target.exists():      # تصادم أسماء داخل نفس الأرشيف
            target = PATHS["raw"] / f"{stem}_{k}{suffix}"
            k += 1
        with zf.open(m) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)
        n_copied += 1
    print(f"استُخرج {n_copied} ملفًا جديدًا، تُخطي {n_skipped} (موجودة مسبقًا)")

RAW_FILES = sorted(p for p in PATHS["raw"].iterdir()
                   if p.suffix.lower() in AUDIO_EXT and p.stat().st_size > 1024)
print(f"\nجرد الخام: {len(RAW_FILES)} ملفًا، "
      f"{sum(p.stat().st_size for p in RAW_FILES)/1e6:.1f} MB إجمالًا")

## 3 — المرحلة 1: الجرد وفك الترميز الموحّد

**الهدف:** تحويل كل صيغة دخل (ogg/mp3/wav...) إلى تمثيل قياسي واحد: **float32، أحادي،
بمعدل عينة أصلي**، محفوظ في `work/01_decoded/aNNN.wav` بمعرّف مستقر — كل المراحل
اللاحقة تُخاطب الملفات بالمعرّف لا بالاسم الأصلي، فلا تنكسر السلسلة عند تغيّر الأسماء.

**فك الترميز:** الأولوية لـ libsndfile (عبر librosa)؛ عند الفشل تسقط إلى ffmpeg عبر
سطر أوامر مؤقت. **quick_test:** يقتصر على `quick_test_files` ملفات (الأقصر أولًا
للأمان الزمني) مقطوعة إلى `quick_test_seconds` — مع تحذير مطبوع بوضوح.

In [ ]:
def decode_any(path):
    """فك ترميز أي صيغة إلى (float32 أحادي, معدل العينة). يعود إلى ffmpeg عند الفشل."""
    try:
        y, sr = librosa.load(str(path), sr=None, mono=True)
        if len(y) > 0:
            return y.astype(np.float32), int(sr)
    except Exception as e:
        LOGGER.warning("فك librosa فشل لـ %s: %s — تجربة ffmpeg",
                       Path(path).name, e)
    tmp = PATHS["work"] / "_decode" / (Path(path).stem + ".wav")
    tmp.parent.mkdir(parents=True, exist_ok=True)
    import subprocess as _sp
    r = _sp.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(path),
                 "-ac", "1", "-ar", "48000", str(tmp)], capture_output=True)
    if r.returncode != 0 or not tmp.exists():
        raise RuntimeError(f"فشل فك الترميز: {Path(path).name}: "
                           f"{r.stderr.decode()[:200]}")
    y, sr = librosa.load(str(tmp), sr=None, mono=True)
    tmp.unlink(missing_ok=True)
    return y.astype(np.float32), int(sr)

selected = RAW_FILES
if CONFIG["quick_test"]:
    infos = []
    for p in RAW_FILES:
        try:
            y_probe, sr_probe = decode_any(p)
            infos.append((p, len(y_probe) / sr_probe))
        except Exception as e:
            LOGGER.warning("تعذر قراءة %s: %s", p.name, e)
    infos.sort(key=lambda t: t[1])
    selected = [p for p, _ in infos[: CONFIG["quick_test_files"]]]
    print(f"!! وضع التحقق السريع: أول {len(selected)} ملف (الأقصر) فقط "
          f"مقطوعة إلى {CONFIG['quick_test_seconds']} ثانية !!")

rows, dec_dir = [], PATHS["work"] / "01_decoded"
dec_dir.mkdir(parents=True, exist_ok=True)
for p in tqdm(selected, desc="فك الترميز", unit="ملف"):
    y, sr = decode_any(p)
    if CONFIG["quick_test"]:
        y = y[: int(sr * CONFIG["quick_test_seconds"])]
    try:
        ch = sf.info(str(p)).channels
    except Exception:
        ch = 1
    fid = f"a{len(rows):03d}"
    wav_path = atomic_write_audio(dec_dir / f"{fid}.wav", y, sr, subtype="FLOAT")
    rows.append({"fid": fid, "orig_name": p.name, "orig_path": str(p),
                 "wav_path": str(wav_path), "sr": sr,
                 "dur_s": round(len(y) / sr, 2),
                 "orig_channels": ch, "orig_ext": p.suffix.lower()})

REGISTRY = pd.DataFrame(rows)
assert REGISTRY["fid"].is_unique
REGISTRY.to_csv(PATHS["reports"] / "registry.csv", index=False)
print(REGISTRY[["fid", "orig_name", "sr", "dur_s", "orig_channels"]]
      .to_string(index=False))
print(f"\nإجمالي المدة: {REGISTRY['dur_s'].sum()/60:.1f} دقيقة عبر {len(REGISTRY)} ملفًا")

## 4 — المرحلة 2: إزالة التكرارات — البصمات

المجموعات الصوتية الخام كثيرًا ما تحوي ازدواجًا (نسخة مُعاد ترميزها، ملف مدمج يجمع
ملفات أخرى). التكرار في بيانات التدريب يضخّم أجزاء بعينها من الصوت ويحرف النموذج.

**البصمة الأولى — تطابق تام:** MD5 لكامل بايتات الملف؛ عند التطابق يبقى الأعلى جودة
(wav > flac > ogg > mp3) ثم الأسبق معرّفًا.

**البصمة الثانية — مغلف الطاقة:** RMS على شبكة خشنة عند 16kHz، لوغاريتمية معايَرة —
بصمة محصّنة ضد إعادة الترميز وفروق المقياس، قادرة على كشف ملف **مدمج** يحتوي ملفًا
آخر كاملًا بداخله (الحالة التي تفلت من فحص MD5 لأن الترميز اختلف).

In [ ]:
def envelope_fingerprint(wav_path, sr_fp=16000, frame=4096, hop_ms=32.0):
    """بصمة مغلف طاقة معيارية: قصيرة ومحصّنة ضد إعادة الترميز وفروق المقياس."""
    y, _ = librosa.load(str(wav_path), sr=sr_fp, mono=True)
    hop = int(sr_fp * hop_ms / 1000)
    if len(y) < frame + 4 * hop:
        return None, 0.0
    n = (len(y) - frame) // hop + 1
    idx = np.arange(frame)[None, :] + hop * np.arange(n)[:, None]
    env = np.sqrt((y[idx] ** 2).mean(axis=1))
    env = np.log1p(env * 1000.0)
    env = (env - env.mean()) / (env.std() + 1e-9)
    return env, len(y) / sr_fp

# ---- الطبقة 1: تطابق تام (يبقى الأفضل جودة عند تطابق المحتوى)
md5_best, exact_dups = {}, []
for _, row in (REGISTRY.assign(_r=REGISTRY["orig_ext"].map(QUALITY_RANK))
               .sort_values(["_r", "fid"])).iterrows():
    try:
        h = hashlib.md5(Path(row.orig_path).read_bytes()).hexdigest()
    except OSError as e:
        LOGGER.warning("تعذر قراءة %s للبصمة: %s — تُعامل كغير مكرر",
                       row.orig_name, e)
        continue
    if h in md5_best:
        exact_dups.append((md5_best[h], row.fid))     # (يبقى, يُستبعد)
    else:
        md5_best[h] = row.fid

# ---- الطبقة 2: بناء بصمات الاحتواء
FPS = {}
for _, row in tqdm(REGISTRY.iterrows(), total=len(REGISTRY),
                   desc="بصمات", unit="ملف"):
    try:
        env, dur = envelope_fingerprint(row.wav_path)
    except Exception as e:
        LOGGER.warning("فشلت بصمة %s: %s — تُستبعد من فحص الاحتواء", row.fid, e)
        env, dur = None, 0.0
    if env is not None:
        FPS[row.fid] = (env, dur, row.orig_ext, row.dur_s)

print(f"بصمات مبنية: {len(FPS)}/{len(REGISTRY)} | تكرارات تامة: {len(exact_dups)}")

### 4.1 إزالة التكرارات — فحص الاحتواء وقرار الاستبعاد

**فحص الاحتواء:** ارتباط متقاطع منزلق معياري لبصمة الأقصر فوق بصمة الأطول — يلتقط
أي ملف يحتوي آخر كاملًا داخله. التكلفة O(n²) لكنها على بصمات خشنة (عنصر واحد لكل
ثلث ثانية) فتبقى سريعة حتى مع مئات الملفات.

**قواعد الاستبعاد:** إن كان **الحاوي أقل جودة** من المحتوى (مثل mp3 يدمج ملفات ogg)
يُستبعد الحاوي — محتواه متاح بصيغة أفضل؛ وإلا يُستبعد المحتوى المكرر داخل الأفضل؛ وعند
تساوي الجودة يبقى الأطول ثم الأسبق معرّفًا. كل قرار يُوثَّق بسببًا في `work/reports/dedup.csv`.

In [ ]:
def best_xcorr(short_env, long_env):
    """أقصى ارتباط متقاطع معياري لـ short فوق long مع موضع المطابقة."""
    n, m = len(short_env), len(long_env)
    if n > m:
        return 0.0, 0
    b = np.concatenate([long_env, np.zeros(n)])
    full = np.correlate(b, short_env, mode="valid")
    a_norm = np.sqrt((short_env ** 2).sum()) + 1e-9
    cs = np.concatenate([[0.0], np.cumsum(b * b)])
    win = np.sqrt(np.maximum(cs[n: n + len(full)] - cs[: len(full)], 1e-9)) + 1e-9
    corr = full / (a_norm * win)
    k = int(np.argmax(corr))
    return float(corr[k]), k

containment = []      # (fid_المحتوى, fid_الحاوي, corr)
ids = list(FPS.keys())
for a in tqdm(ids, desc="فحص الاحتواء", unit="ملف"):
    for b in ids:
        if a == b:
            continue
        env_a, dur_a, _, _ = FPS[a]
        env_b, dur_b, _, _ = FPS[b]
        if dur_a <= dur_b * 1.02:          # الأقصر داخل الأطول فقط
            c, _ = best_xcorr(env_a, env_b)
            if c >= 0.90:
                containment.append((a, b, round(c, 3)))

DROP, REASONS = {}, {}
def mark_drop(fid, reason, kept_alt):
    if fid not in DROP:
        DROP[fid] = kept_alt
        REASONS[fid] = reason

exact_pair_members = set()
for keep_f, drop_f in exact_dups:
    mark_drop(drop_f, f"تطابق تام مع {keep_f}", keep_f)
    exact_pair_members.update({keep_f, drop_f})

for a, b, c in containment:
    if a in exact_pair_members and b in exact_pair_members:
        continue    # زوج متطابق تامًا — حسمته طبقة التطابق أعلاه؛ لا ازدواج قرار
    qa_, qb_ = QUALITY_RANK.get(FPS[a][2], 9), QUALITY_RANK.get(FPS[b][2], 9)
    if qb_ > qa_:
        mark_drop(b, f"حاوي أقل جودة (يضم {a}، ارتباط {c})", a)
    elif qb_ < qa_:
        mark_drop(a, f"محتوى مكرر داخل {b} الأفضل جودة (ارتباط {c})", b)
    else:   # جودة متساوية: يبقى الأطول ثم الأسبق معرّفًا
        keep, drop = (a, b) if (FPS[a][3], a) >= (FPS[b][3], b) else (b, a)
        mark_drop(drop, f"مكرر مقابل {keep} (ارتباط {c})", keep)

### 4.2 التطبيق النهائي وتقرير إزالة التكرارات

تطبيق قرارات الاستبعاد على السجل، مع **شبكة أمان** تمنع حالة نادرة: أن يشير زوجان
متبادلان كلاهما إلى الآخر فيُستبعدان معًا (يُعاد إبقاء الأسبق معرّفًا)، ثم بناء جدول
`KEPT` النهائي مع تقرير موثّق بكل قرار وسببه (يُحفظ في `work/reports/dedup.csv`)،
وإعلان صافي مدة التدريب الحقيقية.

In [ ]:
# شبكة أمان: لا يُستبعد طرفا زوج متبادل كلاهما أبدًا — يُعاد إبقاء الأسبق
changed = True
while changed:
    changed = False
    for drop_f, keep_f in list(DROP.items()):
        if DROP.get(keep_f) == drop_f:      # إشارة متبادلة
            better = min(drop_f, keep_f)
            del DROP[better]
            REASONS[better] = (f"أعيد إبقاؤه — كان مزدوجًا مع "
                               f"{max(drop_f, keep_f)}")
            changed = True

KEPT = REGISTRY[~REGISTRY["fid"].isin(DROP)].reset_index(drop=True)

### 4.3 تقرير إزالة التكرارات

جدول القرارات النهائي — كل استبعاد موثّق بسببه الكامل (يُحفظ في
`work/reports/dedup.csv`) — مع إعلان صافي مدة التدريب الحقيقية بعد التنقية.

In [ ]:
DEDUP_LOG = pd.DataFrame([
    {"fid": row.fid, "orig_name": row.orig_name,
     "action": "استبعاد" if row.fid in DROP else "إبقاء",
     "reason": REASONS.get(row.fid, "-")}
    for _, row in REGISTRY.iterrows()])
DEDUP_LOG.to_csv(PATHS["reports"] / "dedup.csv", index=False,
                 encoding="utf-8-sig")

print(DEDUP_LOG.to_string(index=False))
print(f"\nنتيجة إزالة التكرار: {len(KEPT)} يبقى / {len(DROP)} يُستبعد "
      f"من أصل {len(REGISTRY)}")
print(f"صافي المدة للتدريب: {KEPT['dur_s'].sum()/60:.1f} دقيقة")

## 5 — المرحلة 3: التشخيص — دوال القياس

**الهدف:** مقاييس موضوعية لكل ملف قبل أي تعديل — تحدد قرارات الخط تلقائيًا، وتُستخدم
لاحقًا للمقارنة قبل/بعد لإثبات التحسن بالأرقام لا بالانطباع.

**المقاييس:** المدة، القمة (dBFS)، RMS، **الجهارة المدمجة LUFS** (معيار EBU R128 عبر
pyloudnorm — يحاكي إدراك الأذن)، **أرضية الضوضاء** (المئين العاشر لطاقة الإطارات)،
نسبة عينات القص الرقمي، انزياح DC، ونسبة الكلام عبر **Silero VAD v5** (أدق كاشف نشاط
صوتي مفتوح المصدر؛ مدرّب على 6000+ لغة ويدعم 48kHz مباشرة). النموذج يُحمَّل مرة واحدة
لكل جلسة. الملف الأقصر من نصف ثانية يسقط إلى معايرة RMS بديلًا عن LUFS.

In [ ]:
_VAD_MODEL = None
def vad_model():
    """تحميل نموذج Silero VAD مرة واحدة لكل جلسة."""
    global _VAD_MODEL
    if _VAD_MODEL is None:
        _VAD_MODEL = load_silero_vad()
    return _VAD_MODEL

def speech_spans_silero(y, sr):
    """فواصل الكلام بالثواني عبر Silero VAD (مع إعادة عينة داخلية إلى 16kHz
    عندما يكون معدل الملف خارج المعدلات المدعومة مثل 44100)."""
    y_vad, sr_vad = y, int(sr)
    if sr_vad not in (8000, 16000, 32000, 48000):
        y_vad = soxr.resample(np.asarray(y, dtype=np.float64),
                              sr_vad, 16000, quality="HQ")
        sr_vad = 16000
    wav = _torch.from_numpy(np.ascontiguousarray(y_vad, dtype=np.float32))
    ts = get_speech_timestamps(
        wav, vad_model(), sampling_rate=sr_vad,
        threshold=CONFIG["vad_threshold"],
        min_speech_duration_ms=CONFIG["vad_min_speech_ms"],
        min_silence_duration_ms=100, return_seconds=True)
    return [(t["start"], t["end"]) for t in ts]

def noise_floor_db(y, sr, frame=2048, hop=512):
    """أرضية الضوضاء: المئين العاشر من RMS الإطارات (دسيبل)."""
    if len(y) < frame:
        return db(rms(y))
    n = (len(y) - frame) // hop + 1
    idx = np.arange(frame)[None, :] + hop * np.arange(n)[:, None]
    fr = np.sqrt((y[idx] ** 2).mean(axis=1))
    return db(float(np.percentile(fr, 10)))

def integrated_lufs(y, sr):
    """الجهارة المدمجة LUFS، مع بديل RMS للمقاطع القصيرة/الصامتة جدًا."""
    try:
        meter = pyln.Meter(int(sr))
        return float(meter.integrated_loudness(np.asarray(y, dtype=np.float64)))
    except Exception:
        return rms_db(y)

def diagnose(wav_path):
    """تشخيص كامل لملف واحد. يرجع قاموس مقاييس، أو None عند فشل القراءة."""
    try:
        y, sr = librosa.load(str(wav_path), sr=None, mono=True)
        y = y.astype(np.float32)
    except Exception as e:
        LOGGER.warning("تعذر تشخيص %s: %s", wav_path, e)
        return None
    d = {"dur_s": round(len(y) / sr, 2), "sr": sr,
         "peak_db": round(db(np.max(np.abs(y)) if len(y) else 0.0), 1),
         "rms_db": round(rms_db(y), 1),
         "lufs": round(integrated_lufs(y, sr), 1),
         "floor_db": round(noise_floor_db(y, sr), 1),
         "floor_gap_db": round(rms_db(y) - noise_floor_db(y, sr), 1),
         "clip_pct": round(float(np.mean(np.abs(y) >= 0.99) * 100), 3),
         "dc_offset": round(float(np.mean(y)), 5)}
    if HAS_SILERO:
        spans = speech_spans_silero(y, sr)
        d["speech_pct"] = round(
            sum(e - s for s, e in spans) / max(d["dur_s"], 1e-9) * 100, 1)
    else:
        d["speech_pct"] = None
    return d

### 5.1 تشغيل التشخيص وتقرير ما قبل المعالجة

يُشغَّل التشخيص على كل ملف متبقي بعد إزالة التكرارات ويُحفظ في
`work/reports/diagnostics_pre.csv` — هذا التقرير هو «صورة ما قبل» التي ستُقاس عليها
نتائج المعالجة لاحقًا، ويُبرز فورًا الملفات ذات القص الرقمي أو الأرضية المرتفعة.

In [ ]:
DIAG = []
for _, row in tqdm(KEPT.iterrows(), total=len(KEPT), desc="تشخيص", unit="ملف"):
    d = diagnose(row.wav_path)
    if d is None:
        continue
    d.update({"fid": row.fid, "orig_name": row.orig_name})
    DIAG.append(d)
DIAG = pd.DataFrame(DIAG)
DIAG = DIAG[["fid", "orig_name"] + [c for c in DIAG.columns
                                    if c not in ("fid", "orig_name")]]
DIAG.to_csv(PATHS["reports"] / "diagnostics_pre.csv", index=False,
            encoding="utf-8-sig")

n_clip = int((DIAG["clip_pct"] > 0.05).sum())
print(DIAG.to_string(index=False))
print(f"\nملخص: جهارة متوسطة {DIAG['lufs'].mean():.1f} LUFS | "
      f"أرضية ضوضاء متوسطة {DIAG['floor_db'].mean():.1f} dB | "
      f"ملفات بها قص رقمي: {n_clip}")
if not HAS_SILERO:
    print("تحذير: silero-vad غير متاح — نسبة الكلام غير محسوبة والقص لاحقًا أضعف.")

### 5.2 الرسوم التشخيصية

توزيعات المدة/الجهارة/الأرضية/القمم دفعة واحدة لكشف الشواذ قبل المعالجة. نسخة PNG
تُحفظ في `output/rvc_dataset/qa/`. (تسميات المحاور بالإنجليزية التقنية لضمان سلامة
العرض في كل البيئات.)

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    fig.suptitle("Pre-processing diagnostics", fontsize=14)
    axes[0, 0].hist(DIAG["dur_s"], bins=min(20, max(6, len(DIAG))), color="#4C72B0")
    axes[0, 0].set_title("Duration (s)")
    axes[1, 0].hist(DIAG["floor_db"], bins=15, color="#C44E52")
    axes[1, 0].set_title("Noise floor (dB)")
    axes[0, 1].hist(DIAG["lufs"], bins=15, color="#55A868")
    axes[0, 1].set_title("Integrated loudness (LUFS)")
    axes[1, 1].hist(DIAG["peak_db"], bins=15, color="#8172B2")
    axes[1, 1].axvline(CONFIG["peak_ceiling_db"], ls="--", c="k",
                       label=f"ceiling {CONFIG['peak_ceiling_db']} dB")
    axes[1, 1].set_title("True peak (dBFS)")
    axes[1, 1].legend()
    fig.savefig(PATHS["out_qa"] / "diagnostics_pre.png", dpi=120)
    plt.show()
else:
    print("matplotlib غير متوفر — تخطي الرسوم")

## 6 — المرحلة 4: عزل الصوت البشري — أدوات الفصل

**لماذا؟** أي موسيقى/خلفية في بيانات تدريب RVC تُنقل مباشرة إلى النموذج المستنسَخ.
أقوى نماذج فصل الغناء المتاحة (عبر حزمة `audio-separator` التي تغلف نماذج UVR):

1. **BS-RoFormer** (`ep_368`) — الأعلى في قياسات SDR لاستخراج الغناء.
2. **MelBand RoFormer** — الواجهة الافتراضية الشهيرة والموثوقة.
3. **UVR-MDX-NET-Inst_HQ_3** — MDX-Net الأسرع والأخف (احتياط).

**هذه الخلية تجهّز الدوال فقط:** تهيئة آمنة لكتم سجلات المكتبة، ذاكرة نماذج مشتركة،
وحل مؤشرات المسار النظيف من أسماء مخرجات النماذج (مثل «(Vocals)» مقابل
«(Instrumental)») — النماذج ترجع أسماء ملفات وقد تكون نسبية أو مطلقة حسب الإصدار.

In [ ]:
SEP_DIR = PATHS["work"] / "04_vocals"
SEP_DIR.mkdir(parents=True, exist_ok=True)

def make_separator(out_dir):
    """تهيئة Separator مع كتم سجلاته؛ ذاكرة نماذج مشتركة لتجنب إعادة التنزيل."""
    try:
        return Separator(output_dir=str(out_dir),
                         model_file_dir=str(PATHS["models"]),
                         log_level=logging.WARNING)
    except TypeError:       # إصدارات أقدم بلا log_level
        return Separator(output_dir=str(out_dir),
                         model_file_dir=str(PATHS["models"]))

def resolve_sep_outputs(outs, out_dir):
    """مسارات مطلقة من مخرجات Separator (قد تعود نسبية أو مطلقة حسب الإصدار)."""
    return [p if p.is_absolute() else Path(out_dir) / p.name
            for p in map(Path, outs)]

def pick_stem(paths, positive, negative):
    """اختيار المسار النظيف: أول مطابقة للمؤشرات الإيجابية، وإلا غير السلبية."""
    low = {p: p.name.lower() for p in paths}
    for p, name in low.items():
        if any(k in name for k in positive):
            return p
    for p, name in low.items():
        if not any(k in name for k in negative):
            return p
    return paths[0] if paths else None

def separate_to_file(wav_in, model_name, out_dir, positive, negative):
    """فصل بملف نموذج واحد وإرجاع مسار المسار النظيف (أو None عند الفشل)."""
    sep = make_separator(out_dir)
    sep.load_model(model_filename=model_name)
    outs = sep.separate(str(wav_in))
    return pick_stem(resolve_sep_outputs(outs, out_dir), positive, negative)

print("أدوات الفصل جاهزة:", "متاحة" if HAS_ASEP else "غير متاحة — المرحلة ستُتخطى بأمان")

### 6.1 قرار الفصل: الفحص الاستباقي الحقيقي (وضع auto)

**قياس لا تخمين.** قواعد الكشف اليدوية عن الموسيقى تنخدع بالكلام المتنغّم (نغمات
الكلام تُشبه النغمات الموسيقية في أي محفز طيفي بسيط)، لذا يقيس الدفتر الموسيقى
**بنموذج فصل حقيقي**: نموذج خفيف وسريع (`4_HP-Vocal-UVR`) على عينة 20 ثانية من منتصف
كل ملف، ثم يقيس الفارق بين طاقة مسار «الآلات» ومسار «الغناء» بالدسيبل:

- فارق ≈ −50 إلى −60 dB (صمت رقمي) → **لا موسيقى** → لا فصل. (النطاق المقيس فعليًا
  على هذا الأرشيف: −48 إلى −62 dB — رسائل صوتية نقية.)
- فارق > `probe_flag_db` (−25 dB) → موسيقى فعلية → **فصل كامل** بأفضل نموذج.

`"all"` يفصل الجميع قسرًا، و`"none"` يعطّل، وقائمة معرفات تفرض ملفات بعينها.

In [ ]:
def probe_music(wav_path, fid):
    """فحص استباقي حقيقي: نسبة دسيبل (آلات/غناء) على عينة من منتصف الملف."""
    try:
        y, sr = librosa.load(str(wav_path), sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر تحميل %s للفحص: %s", wav_path, e)
        return None, None
    n_take = min(int(sr * CONFIG["probe_seconds"]), len(y))
    start = max(0, (len(y) - n_take) // 2)
    seg = y[start: start + n_take]
    pdir = PATHS["work"] / "_probe" / fid
    pdir.mkdir(parents=True, exist_ok=True)
    probe_in = atomic_write_audio(pdir / "probe_in.wav", seg, sr)
    clean = separate_to_file(probe_in, CONFIG["probe_model"], pdir,
                             positive=["vocals"],
                             negative=["instrumental", "inst"])
    if clean is None:
        safe_rmtree(pdir)
        return None, None
    y_voc, _ = librosa.load(str(clean), sr=None, mono=True)
    instr = [p for p in pdir.glob("*.wav")
             if p != clean and p != probe_in]
    if not instr:
        safe_rmtree(pdir)
        return None, None
    y_ins, _ = librosa.load(str(instr[0]), sr=None, mono=True)
    ratio = rms_db(y_ins) - rms_db(y_voc)
    safe_rmtree(pdir)             # تنظيف مساحة الفحص
    return ratio, clean

### 6.1.1 قرار الفصل: الوضع والفحص الاستباقي

تحديد قائمة الأهداف حسب `separation_mode`: `"auto"` يشغل الفحص الاستباقي الحقيقي
على كل ملف (مع حماية فردية — فشل فحص ملف لا يوقف الخط)، و`"all"` يفرض الفصل على
الجميع، و`"none"` يعطّل، وقائمة معرفات تفرض ملفات بعينها. النتائج الكاملة تُطبع
جدولًا شفافًا وتُوثّق في البيان.

In [ ]:
mode = CONFIG["separation_mode"]
MUSIC_FLAGS = {}
if not HAS_ASEP:
    print("audio-separator غير متوفر — تعطيل مرحلة الفصل (موثّق في البيان).")
    targets = []
elif mode == "none":
    print("مرحلة الفصل معطلة بطلب المستخدم (separation_mode=none).")
    targets = []
elif mode == "all":
    targets = list(KEPT["fid"])
    print(f"الفصل إلزامي لكل الملفات: {len(targets)}")
elif isinstance(mode, (list, tuple)):
    targets = [f for f in mode if f in set(KEPT["fid"])]
    print(f"الفصل لقائمة محددة: {targets}")
else:   # auto — القياس الفعلي
    targets = []
    print("الفحص الاستباقي (نموذج حقيقي على عينة 20 ثانية لكل ملف):")
    probe_rows = []
    for _, row in KEPT.iterrows():
        try:
            ratio, _ = probe_music(row.wav_path, row.fid)
        except Exception as e:
            LOGGER.warning("الفحص الاستباقي فشل لـ %s: %s", row.fid, e)
            ratio = None
        flag = (ratio is not None) and (ratio > CONFIG["probe_flag_db"])
        MUSIC_FLAGS[row.fid] = {
            "inst_over_vocals_db": None if ratio is None else round(ratio, 1),
            "music_detected": bool(flag)}
        probe_rows.append((row.fid, row.orig_name,
                           "-" if ratio is None else f"{ratio:.1f} dB", flag))
        if flag:
            targets.append(row.fid)
    print(f"\n{'fid':<6} {'الملف':<40} {'آلات-غناء':>10}  موسيقى؟")
    print("\n".join(f"{fid:<6} {name:<40} {r:>10}  {'نعم' if flag else 'لا'}"
                    for fid, name, r, flag in probe_rows))
    print(f"\nالفحص انتهى: {len(targets)} ملفًا يحتاج فصلًا فعليًا من أصل {len(KEPT)}")

### 6.2 الفصل الكامل للملفات المعلنة

تشغيل سلسلة النماذج على الملفات المعلنة فقط (مع الذاكرة المؤقتة وسقوط آمن للنموذج
التالي عند فشل أحدها). النتيجة النهائية: خريطة `VOC_PATHS` — لكل ملف معرّف ومساره
بعد الفصل، أو مساره الأصلي إن لم يحتج فصلًا. **ملاحظة أداء صادقة:** على GPU ثوانٍ لكل
ملف؛ على CPU دقائق لكل دقيقة صوت.

In [ ]:
VOC_PATHS = {row.fid: row.wav_path for _, row in KEPT.iterrows()}
SEP_LOG = []
if targets:
    for fid in tqdm(targets, desc="فصل الصوت البشري", unit="ملف"):
        cache = SEP_DIR / f"{fid}.wav"
        used_model = "-"
        if cache.exists() and CONFIG["use_cache"]:
            used_model = "ذاكرة مؤقتة"
        else:
            last_err = None
            for model_name in CONFIG["separation_models"]:
                try:
                    tdir = PATHS["work"] / "_sep_tmp" / fid
                    tdir.mkdir(parents=True, exist_ok=True)
                    clean = separate_to_file(VOC_PATHS[fid], model_name, tdir,
                                             positive=["vocals"],
                                             negative=["instrumental", "inst"])
                    if clean is None or not Path(clean).exists():
                        raise RuntimeError("لم ينتج النموذج مسارًا نظيفًا")
                    y, sr = librosa.load(str(clean), sr=None, mono=True)
                    atomic_write_audio(cache, y.astype(np.float32), sr)
                    safe_rmtree(tdir)
                    used_model = model_name
                    break
                except Exception as e:
                    last_err = e
                    LOGGER.warning("نموذج %s فشل لـ %s: %s", model_name, fid, e)
            else:
                SEP_LOG.append({"fid": fid, "model": f"فشل الجميع: {last_err}",
                                "ok": False})
                print(f"  تحذير: فشلت كل نماذج الفصل لـ {fid} — "
                      f"سيُستخدم الصوت غير المفصول")
                continue
        VOC_PATHS[fid] = str(cache)
        SEP_LOG.append({"fid": fid, "model": used_model,
                        "ok": used_model != "فشل الجميع"})

if SEP_LOG:
    print(pd.DataFrame(SEP_LOG).to_string(index=False))
else:
    print("\nلا حاجة لأي فصل — مسارات الصوت البشري = الصوت الأصلي لكل الملفات.")

## 7 — المرحلة 5: إزالة الضوضاء — الإعداد

**الخيار الأول — `UVR-DeNoise`** (نموذج VR Arch مدرّب خصيصًا للضوضاء): معيار مجتمع
RVC/UVR لتنقية بيانات التدريب؛ يعالج الضوضاء غير الثابتة (شارع، حشد، هواء) دون «صوت
معدني». **البديل — `noisereduce`** (بوابة طيفية): للضوضاء الثابتة فقط ويُستخدم بشدة
احترازية (`prop_decrease=0.75`) حفاظًا على طابع الصوت. الوضع `auto` يختار UVR إن توفر
وإلا البديل وإلا التعطيل — وكل قرار يُطبع صريحًا.

In [ ]:
DEN_DIR = PATHS["work"] / "05_denoised"
DEN_DIR.mkdir(parents=True, exist_ok=True)

engine = CONFIG["denoise_engine"]
if engine == "auto":
    engine = "uvr" if HAS_ASEP else ("noisereduce" if HAS_NR else "none")
if engine == "uvr" and not HAS_ASEP:
    print("audio-separator غير متوفر — تساقط إلى noisereduce")
    engine = "noisereduce" if HAS_NR else "none"
if engine == "noisereduce" and not HAS_NR:
    engine = "none"
print("محرك إزالة الضوضاء:", engine)

### 7.1 إزالة الضوضاء — التنفيذ والتقرير

معالجة كل ملف بالمحرك المختار (مع الذاكرة المؤقتة)، و**قياس أرضية الضوضاء قبل/بعد
لكل ملف** — التحسن يُطبع بالدسيبل ويُحفظ في `work/reports/denoise_report.csv`:
برهان رقمي أن التنقية أضافت قيمة. الفشل الفردي لا يوقف الخط: الملف يمرر مباشرة مع
تحذير موثّق.

In [ ]:
DEN_PATHS, DEN_REPORT = {}, []
for _, row in tqdm(KEPT.iterrows(), total=len(KEPT),
                   desc="إزالة الضوضاء", unit="ملف"):
    fid = row.fid
    src = VOC_PATHS[fid]
    cache = DEN_DIR / f"{fid}.wav"
    try:
        y, sr = librosa.load(src, sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر تحميل %s: %s — تمرير مباشر", fid, e)
        DEN_PATHS[fid] = src
        continue
    floor_before = noise_floor_db(y, sr)
    if engine == "none":
        DEN_PATHS[fid] = src
        DEN_REPORT.append({"fid": fid, "engine": "معطل",
                            "floor_before_db": round(floor_before, 1),
                            "floor_after_db": round(floor_before, 1),
                            "improvement_db": 0.0})
        continue
    if cache.exists() and CONFIG["use_cache"]:
        y2, sr2 = librosa.load(str(cache), sr=None, mono=True)
    else:
        try:
            if engine == "uvr":
                tdir = PATHS["work"] / "_den_tmp" / fid
                tdir.mkdir(parents=True, exist_ok=True)
                clean = separate_to_file(src, CONFIG["denoise_model"], tdir,
                                         positive=["no noise"],
                                         negative=["noise"])
                y2, sr2 = librosa.load(str(clean), sr=None, mono=True)
                safe_rmtree(tdir)
            else:   # noisereduce
                y2 = nr.reduce_noise(y=y.astype(np.float32), sr=sr,
                                     prop_decrease=CONFIG["noisereduce_prop_decrease"],
                                     use_tqdm=False)
                sr2 = sr
            atomic_write_audio(cache, y2.astype(np.float32), sr2)
        except Exception as e:
            LOGGER.warning("إزالة الضوضاء فشلت لـ %s: %s — تمرير مباشر", fid, e)
            print(f"  تحذير: فشلت إزالة الضوضاء لـ {fid} — تمرير مباشر دونها")
            DEN_PATHS[fid] = src
            DEN_REPORT.append({"fid": fid, "engine": "فشل — تمرير مباشر",
                               "floor_before_db": round(floor_before, 1),
                               "floor_after_db": round(floor_before, 1),
                               "improvement_db": 0.0})
            continue
    floor_after = noise_floor_db(y2, sr2)
    DEN_PATHS[fid] = str(cache)
    DEN_REPORT.append({"fid": fid, "engine": engine,
                       "floor_before_db": round(floor_before, 1),
                       "floor_after_db": round(floor_after, 1),
                       "improvement_db": round(floor_before - floor_after, 1)})

### 7.2 تقرير إزالة الضوضاء

تجميع التقرير النهائي وحفظه في `work/reports/denoise_report.csv`، مع قراءة صادقة
للأرقام: على مصادر نظيفة أصلًا قد يكون تغير أرضية الضوضاء صفريًا أو سالبًا قليلًا —
المرحلة تبقى مهمة عندما توجد ضوضاء فعلية.

In [ ]:
DEN_REPORT = pd.DataFrame(DEN_REPORT)
DEN_REPORT.to_csv(PATHS["reports"] / "denoise_report.csv",
                  index=False, encoding="utf-8-sig")
print(DEN_REPORT.to_string(index=False))
if len(DEN_REPORT) and engine != "none":
    avg = DEN_REPORT['improvement_db'].mean()
    print(f"\nمتوسط تغير أرضية الضوضاء عند المعالجة: {avg:.1f} dB")
    if avg <= 0:
        print("ملاحظة صادقة: التحسن صفر أو سالب — المصادر نظيفة أصلًا من الضوضاء "
              "الثابتة؛ المرحلة تبقى مهمة عندما توجد ضوضاء فعلية (وسجلت هنا أرقامها "
              "الحقيقية لا أرقامًا متمناة).")

## 8 — المرحلة 6: إزالة الصدى والارتداد (De-reverb / De-echo)

**لماذا؟** ارتداد الغرفة يُلوّث البصمات الطيفية التي يتعلمها النموذج ويجعل الصوت
المستنسَخ «عائمًا». **النموذج:** `UVR-DeEcho-DeReverb` (VR Arch، من FoxJoy) — المعيار
المعتمد في مجتمع UVR لإعداد بيانات RVC.

**حدّة صادقة:** لا يوجد بديل بايثوني خفيف موثوق لهذه المهمة؛ إن غابت حزمة الفصل
تُتخطى المرحلة بأمان مع توثيق ذلك في البيان — لا نتظاهر بتنقية لم تحدث. الفشل الفردي
لا يوقف الخط: الملف يمرر مع تحذير.

In [ ]:
DRV_DIR = PATHS["work"] / "06_dereverb"
DRV_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATHS, DRV_REPORT = {}, []
use_dereverb = CONFIG["dereverb_enable"] and HAS_ASEP
if CONFIG["dereverb_enable"] and not HAS_ASEP:
    print("audio-separator غير متوفر — تخطي إزالة الصدى (موثّق في البيان).")

for _, row in tqdm(KEPT.iterrows(), total=len(KEPT), desc="إزالة الصدى", unit="ملف"):
    fid = row.fid
    src = DEN_PATHS[fid]
    cache = DRV_DIR / f"{fid}.wav"
    if not use_dereverb:
        CLEAN_PATHS[fid] = src
        DRV_REPORT.append({"fid": fid, "status": "متخطاة"})
        continue
    if cache.exists() and CONFIG["use_cache"]:
        CLEAN_PATHS[fid] = str(cache)
        DRV_REPORT.append({"fid": fid, "status": "ذاكرة مؤقتة"})
        continue
    try:
        tdir = PATHS["work"] / "_drv_tmp" / fid
        tdir.mkdir(parents=True, exist_ok=True)
        clean = separate_to_file(src, CONFIG["dereverb_model"], tdir,
                                 positive=["no reverb", "dry"],
                                 negative=["reverb", "echo"])
        y, sr = librosa.load(str(clean), sr=None, mono=True)
        atomic_write_audio(cache, y.astype(np.float32), sr)
        safe_rmtree(tdir)
        CLEAN_PATHS[fid] = str(cache)
        DRV_REPORT.append({"fid": fid, "status": "تمت"})
    except Exception as e:
        LOGGER.warning("إزالة الصدى فشلت لـ %s: %s — تمرير مباشر", fid, e)
        print(f"  تحذير: فشلت إزالة الصدى لـ {fid} — تمرير مباشر")
        CLEAN_PATHS[fid] = src
        DRV_REPORT.append({"fid": fid, "status": "فشل — تمرير مباشر"})

DRV_REPORT = pd.DataFrame(DRV_REPORT)
print(DRV_REPORT.to_string(index=False))
print(f"\nحالة المسارات النهائية: {len(CLEAN_PATHS)} ملفًا نظيفًا جاهزًا للتقطيع")

## 9 — المرحلة 7: القص الذكي — خوارزميات التقطيع

**لماذا نطاق 3–12 ثانية؟** توصيات تدريب RVC الرسمية ومجتمعه: مقاطع قصيرة متجانسة تمنح
فهرس الاسترجاع (indexer) أفضل تطابق، وتجنّب المقاطع الطويلة التي تخفق فيها النماذج أو
تهدر الذاكرة.

**الخوارزمية (لكل ملف):**

1. **Silero VAD v5** يستخرج فواصل الكلام الدقيقة — لا قطع داخل الكلمات أبدًا.
2. **دمج آمن:** الفجوات ≤ `vad_min_gap_ms` تُدمج داخل مقطع واحد؛ ثم المقاطع المتجاورة
   التي تفصلها فجوة ≤ `merge_gap_s` ويظل حاصلها تحت الطول الأقصى تُدمج (وقفة طبيعية
   داخل المقطع أفضل من تقطيع متكسر).
3. **هوامش:** `edge_pad_s` حول كل مقطع حفاظًا على اندفاعات الحروف الساكنة.
4. **تقسيم الطويلة:** ما تجاوز `max_clip_s` يُقسم عند **أهدأ نافذة 300ms** في النطاق
   الأوسط (20%–80%) — نقطة العبور الطبيعية بين الجمل، مع ضمان البقاء داخل المقطع.
5. **التقليم:** ما دون `drop_below_s` يُهمَل (شظايا).

**الفشل الآمن:** غياب silero → قص طاقي (عتبة فوق أرضية الضوضاء بـ 10dB) مع تحذير أن
الدقة أدنى عند حدود الكلام.

In [ ]:
def vad_spans(y, sr):
    """فواصل الكلام (بداية، نهاية) بالثواني — Silero أو البديل الطاقي."""
    if HAS_SILERO:
        return speech_spans_silero(y, sr)
    frame, hop = 2048, int(sr * 0.01)      # بديل طاقي عند غياب silero
    if len(y) < frame:
        return [(0.0, len(y) / sr)] if len(y) else []
    n = (len(y) - frame) // hop + 1
    idx = np.arange(frame)[None, :] + hop * np.arange(n)[:, None]
    fr = np.sqrt((y[idx] ** 2).mean(axis=1))
    floor = float(np.percentile(fr, 10))
    thr = floor * 10 ** (10 / 20)          # أرضية + 10 dB
    active = fr > thr
    spans, start = [], None
    for i, a in enumerate(active):
        if a and start is None:
            start = i * hop / sr
        elif not a and start is not None:
            spans.append((start, i * hop / sr))
            start = None
    if start is not None:
        spans.append((start, len(y) / sr))
    return spans

def merge_small_gaps(spans, max_gap):
    """دمج الفجوات الصغيرة داخل مقطع واحد."""
    if not spans:
        return []
    out = [list(spans[0])]
    for s, e in spans[1:]:
        if s - out[-1][1] <= max_gap:
            out[-1][1] = e
        else:
            out.append([s, e])
    return [tuple(x) for x in out]

def merge_short_neighbors(spans, max_len, max_gap):
    """دمج مقاطع متجاورة ما دام الحاصل (مع الفجوة) ضمن الطول الأقصى."""
    changed = True
    spans = [list(s) for s in spans]
    while changed:
        changed = False
        for i in range(len(spans) - 1):
            s1, e1 = spans[i]
            s2, e2 = spans[i + 1]
            if (s2 - e1) <= max_gap and (e2 - s1) <= max_len:
                spans[i] = [s1, e2]
                del spans[i + 1]
                changed = True
                break
    return [tuple(x) for x in spans]

### 9.0.1 تقسيم المقاطع الطويلة

`split_overlong` تُقسم أي مقطع يجاوز `max_clip_s` عند **أهدأ نافذة 300ms** في النطاق
الأوسط (20%–80%) — نقطة العبور الطبيعية بين الجمل — مع ضمان البقاء داخل المقطع
وتقدم حتمي (لا حلقات لا نهائية).

In [ ]:
def split_overlong(span, y, sr, max_len, probe=0.30):
    """تقسيم مقطع طويل عند أهدأ نافذة وسطية — تكراريًا حتى الاستيفاء."""
    queue, done = [span], []
    while queue:
        s, e = queue.pop()
        if e - s <= max_len:
            done.append((s, e))
            continue
        mid_lo, mid_hi = s + (e - s) * 0.20, e - (e - s) * 0.20
        i0, i1 = int(mid_lo * sr), int(mid_hi * sr)
        seg = y[i0:i1]
        win = int(probe * sr)
        if len(seg) < 2 * win:
            cut = (s + e) / 2          # لا مفر: الشطر في المنتصف
        else:
            n = (len(seg) - win) // (win // 2) + 1
            energies = [float(np.mean(seg[k * (win // 2): k * (win // 2) + win] ** 2))
                        for k in range(n)]
            k = int(np.argmin(energies))
            cut = i0 / sr + (k * (win // 2) + win / 2) / sr
        cut = min(max(cut, s + 0.5), e - 0.5)   # البقاء داخل المقطع
        queue.extend([(s, cut), (cut, e)])
    return done

### 9.1 بناء المقاطع لكل ملف

تطبيق الخوارزمية على كل ملف نظيف (مع حماية فردية: فشل ملف لا يوقف الخط)، والنتيجة
جدول `SEGMENTS` يوثّق كل مقطع بمعرّفه وترتيبه وبدايته ونهايته ومدته — يُحفظ في
`work/reports/segments.csv`.

In [ ]:
seg_rows = []
for _, row in tqdm(KEPT.iterrows(), total=len(KEPT), desc="تقطيع", unit="ملف"):
    fid = row.fid
    try:
        y, sr = librosa.load(CLEAN_PATHS[fid], sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر تحميل %s للتقطيع: %s — تخطي", fid, e)
        continue
    pad = CONFIG["edge_pad_s"]
    raw = vad_spans(y, sr)
    raw = merge_small_gaps(raw, CONFIG["vad_min_gap_ms"] / 1000.0)
    raw = merge_short_neighbors(raw, CONFIG["max_clip_s"], CONFIG["merge_gap_s"])
    raw = [(max(0.0, s - pad), min(len(y) / sr, e + pad)) for s, e in raw]
    spans = []
    for s, e in raw:
        if e - s > CONFIG["max_clip_s"]:
            spans.extend(split_overlong((s, e), y, sr, CONFIG["max_clip_s"]))
        else:
            spans.append((s, e))
    final = [(s, e) for s, e in spans if e - s >= CONFIG["drop_below_s"]]
    final.sort()                       # ترتيب زمني صارم لترقيم المقاطع
    seg_rows.extend({"fid": fid, "clip_idx": idx,
                     "start_s": round(s, 3), "end_s": round(e, 3),
                     "dur_s": round(e - s, 3)}
                    for idx, (s, e) in enumerate(final))

if not HAS_SILERO:
    print("تحذير: القص طاقي (silero-vad غير متوفر) — دقة أقل عند حدود الكلام.")

SEGMENTS = pd.DataFrame(seg_rows)
assert SEGMENTS["dur_s"].between(0, CONFIG["max_clip_s"] + 0.01).all()
SEGMENTS.to_csv(PATHS["reports"] / "segments.csv", index=False,
                encoding="utf-8-sig")

print(SEGMENTS.head(15).to_string(index=False))
if len(SEGMENTS):
    print(f"\nمقاطع: {len(SEGMENTS)} | متوسط {SEGMENTS['dur_s'].mean():.1f} ث | "
          f"وسيط {SEGMENTS['dur_s'].median():.1f} ث | "
          f"مدى [{SEGMENTS['dur_s'].min():.1f}, {SEGMENTS['dur_s'].max():.1f}] ث")
    print(f"إجمالي كلام نظيف: {SEGMENTS['dur_s'].sum()/60:.1f} دقيقة")
else:
    print("تحذير: لا مقاطع — تحقق من عتبة VAD أو جودة الصوت")

## 10 — المرحلة 8: توحيد الجهارة وحماية القمم

**الطريقة:** قياس **الجهارة المدمجة** وفق معيار البث EBU R128 عبر `pyloudnorm`، ثم كسب
موحد إلى `target_lufs` لكل مقطع — التجانس بين المقاطع أهم من القيمة المطلقة لنماذج
التدريب. بعدها سقف قمة آمن `peak_ceiling_db` بحكم قاسٍ مع توثيق أي حالات تقييد.

**تفاصيل:** إزالة انزياح DC قبل القياس؛ المقطع الأقصر من نصف ثانية يسقط إلى معايرة RMS؛
الملفات الكاملة تُوحَّد بنفس المنطق لشجرة `full_clean`.

In [ ]:
NORM_CLIPS_DIR = PATHS["work"] / "07_clips"
NORM_FULL_DIR = PATHS["work"] / "07_full"
NORM_CLIPS_DIR.mkdir(parents=True, exist_ok=True)
NORM_FULL_DIR.mkdir(parents=True, exist_ok=True)

def normalize_audio(y, sr):
    """إزالة DC ثم توحيد LUFS ثم سقف قمة. ترجع (y المهيأ, معلومات)."""
    y = y - float(np.mean(y))
    lufs = integrated_lufs(y, sr)
    info = {"lufs_before": lufs}
    ceiling = 10 ** (CONFIG["peak_ceiling_db"] / 20.0)
    if np.isfinite(lufs):
        gain = 10 ** ((CONFIG["target_lufs"] - lufs) / 20.0)
    else:
        gain = 10 ** ((CONFIG["target_lufs"] - rms_db(y)) / 20.0)
    y = y * gain
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    if peak > ceiling:
        y = y * (ceiling / peak)
        info["peak_limited"] = True
    else:
        info["peak_limited"] = False
    info["lufs_after"] = integrated_lufs(y, sr)
    info["peak_after_db"] = db(float(np.max(np.abs(y))) if len(y) else 0.0)
    return y.astype(np.float32), info

clip_rows = []
for fid, segs in tqdm(SEGMENTS.groupby("fid"), total=SEGMENTS["fid"].nunique(),
                      desc="توحيد الجهارة", unit="ملف"):
    try:
        y, sr = librosa.load(CLEAN_PATHS[fid], sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر تحميل %s للتوحيد: %s — تخطي مقاطعه", fid, e)
        continue
    for _, seg in segs.iterrows():
        i0, i1 = int(seg.start_s * sr), int(seg.end_s * sr)
        y_c, info = normalize_audio(y[i0:i1], sr)
        name = f"{fid}_c{int(seg.clip_idx):03d}.wav"
        atomic_write_audio(NORM_CLIPS_DIR / name, y_c, sr, subtype="FLOAT")
        clip_rows.append({"clip_name": name, "fid": fid,
                          "clip_idx": int(seg.clip_idx), "dur_s": seg.dur_s,
                          "lufs_after": round(info["lufs_after"], 1),
                          "peak_db": round(info["peak_after_db"], 1)})
CLIPS = pd.DataFrame(clip_rows)

full_rows = []
for _, row in tqdm(KEPT.iterrows(), total=len(KEPT), desc="توحيد الملفات الكاملة",
                   unit="ملف"):
    fid = row.fid
    try:
        y, sr = librosa.load(CLEAN_PATHS[fid], sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر تحميل %s: %s — تخطي", fid, e)
        continue
    y, info = normalize_audio(y, sr)
    atomic_write_audio(NORM_FULL_DIR / f"{fid}_full.wav", y, sr, subtype="FLOAT")
    full_rows.append({"fid": fid, "full_name": f"{fid}_full.wav",
                      "dur_s": row.dur_s,
                      "lufs_after": round(info["lufs_after"], 1)})
FULLS = pd.DataFrame(full_rows)

print("مقاطع موحدة:", len(CLIPS), "| جهارة نهائية: متوسط "
      f"{CLIPS['lufs_after'].mean():.1f} LUFS (الهدف {CONFIG['target_lufs']})")
print(CLIPS.head(10).to_string(index=False))

## 11 — المرحلة 9: إعادة العينة والتصدير النهائي

**معايير RVC v2:** أحادي القناة، **48kHz** (أعلى معدل يدعمه)، **WAV PCM 16-bit** — عبر
**`soxr` بجودة VHQ** (أفضل محرك إعادة عينة مفتوح المصدر في كل المقاييس المنشورة؛ يُفضَّل
على scipy-resample وresampy). نماذج VR Arch تُخرج 44.1kHz أحيانًا — تُوحَّد كلها هنا.

**شجرتا المخرجات:** `output/rvc_dataset/clips/` (مقاطع 3–12 ث جاهزة للتدريب مباشرة —
المسار الموصى به) و`output/rvc_dataset/full_clean/` (الملفات الكاملة المنظفة لمن يفضّل
ترك التقطيع لواجهة RVC نفسها). حارس قمة أخير قبل تحويل 16-bit لمنع أي لفّ عند التقريب،
وكتابة ذرّية، وحماية فردية لكل ملف.

In [ ]:
TARGET_SR = int(CONFIG["target_sr"])
N_EXPORTED = {"clips": 0, "clips_failed": 0, "full": 0, "full_failed": 0}

def export_final(src_path, dest_path):
    """إعادة عينة VHQ إلى target_sr ثم كتابة PCM_16 بحارس قمة وذرّية."""
    y, sr = librosa.load(str(src_path), sr=None, mono=True)
    if sr != TARGET_SR:
        y = soxr.resample(y.astype(np.float64), sr, TARGET_SR, quality="VHQ")
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    if peak > 0.999:
        y = y * (0.999 / peak)
    atomic_write_audio(dest_path, y.astype(np.float32), TARGET_SR,
                       subtype="PCM_16")

for _, r in tqdm(CLIPS.iterrows(), total=len(CLIPS), desc="تصدير المقاطع",
                 unit="مقطع"):
    try:
        export_final(NORM_CLIPS_DIR / r.clip_name,
                     PATHS["out_clips"] / r.clip_name)
        N_EXPORTED["clips"] += 1
    except Exception as e:
        N_EXPORTED["clips_failed"] += 1
        LOGGER.warning("فشل تصدير المقطع %s: %s", r.clip_name, e)

for _, r in tqdm(FULLS.iterrows(), total=len(FULLS), desc="تصدير الملفات الكاملة",
                 unit="ملف"):
    try:
        export_final(NORM_FULL_DIR / r.full_name,
                     PATHS["out_full"] / r.full_name)
        N_EXPORTED["full"] += 1
    except Exception as e:
        N_EXPORTED["full_failed"] += 1
        LOGGER.warning("فشل تصدير %s: %s", r.full_name, e)

probe_ok = 0
for p in sorted(PATHS["out_clips"].glob("*.wav"))[:5]:
    info = sf.info(str(p))
    assert info.samplerate == TARGET_SR and info.channels == 1 \
        and info.subtype == "PCM_16", f"مواصفات خاطئة: {p}"
    probe_ok += 1
print(f"تم تصدير {N_EXPORTED['clips']} مقطعًا و{N_EXPORTED['full']} ملفًا كاملًا "
      f"(فشل: {N_EXPORTED['clips_failed']}/{N_EXPORTED['full_failed']}).")
print(f"تحقق عينة: {probe_ok} ملفات — كلها 48kHz/mono/PCM_16 كما هو مطلوب.")

## 12 — المرحلة 10: فحص الجودة النهائي والمقارنة قبل/بعد

**التحقق الموضوعي لا الانطباعي.** لكل مقطع نهائي يُعاد قياس كل شيء **من الملف المكتوب
فعليًا على القرص** (لا من الذاكرة): المواصفات (48k/mono/PCM_16)، المدة ضمن النطاق،
الجهارة حول الهدف، القمة تحت السقف، لا قص رقمي. ثم المقارنة الكمية قبل/بعد لكل ملف
أصلي: تحسن أرضية الضوضاء بالدسيبل، ونسبة الكلام المستبقاة — البرهان الرقمي أن
المعالجة أضافت قيمة دون إتلاف المحتوى.

In [ ]:
qa_rows = []
for _, r in tqdm(CLIPS.iterrows(), total=len(CLIPS), desc="فحص الجودة", unit="مقطع"):
    p = PATHS["out_clips"] / r.clip_name
    try:
        info = sf.info(str(p))
        y, sr = librosa.load(str(p), sr=None, mono=True)
    except Exception as e:
        LOGGER.warning("تعذر فحص %s: %s", r.clip_name, e)
        continue
    qa_rows.append({
        "clip_name": r.clip_name, "fid": r.fid,
        "sr": info.samplerate, "channels": info.channels,
        "subtype": info.subtype, "dur_s": round(len(y) / sr, 2),
        "lufs": round(integrated_lufs(y, sr), 1),
        "peak_db": round(db(np.max(np.abs(y))), 1),
        "floor_db": round(noise_floor_db(y, sr), 1),
        "clip_pct": round(float(np.mean(np.abs(y) >= 0.99) * 100), 3)})
QA = pd.DataFrame(qa_rows)
QA["check_sr"] = QA["sr"] == TARGET_SR
QA["check_mono"] = QA["channels"] == 1
QA["check_16bit"] = QA["subtype"] == "PCM_16"
QA["check_dur"] = QA["dur_s"].between(CONFIG["drop_below_s"],
                                      CONFIG["max_clip_s"] + 0.02)
QA["check_lufs"] = (QA["lufs"] - CONFIG["target_lufs"]).abs() <= 2.0
QA["check_peak"] = QA["peak_db"] <= CONFIG["peak_ceiling_db"] + 0.5
QA["check_noclip"] = QA["clip_pct"] <= 0.05
CHECKS = [c for c in QA.columns if c.startswith("check_")]
QA["pass_all"] = QA[CHECKS].all(axis=1)
QA.to_csv(PATHS["out_qa"] / "clips_qa.csv", index=False, encoding="utf-8-sig")

### 12.1 المقارنة قبل/بعد لكل ملف أصلي

**مقياس ثابت الجهارة:** مقارنة أرضية الضوضاء المطلقة قبل/بعد تكون غير عادلة لأن توحيد
الجهارة يرفع كل شيء معًا؛ لذا نقيس **فجوة الأرضية تحت مستوى الكلام** (RMS − أرضية،
بالدسيبل) على الملف الكامل النهائي مقابل الأصل — مقياس لا يتأثر بتغيير الجهارة ويعكس
نسبة الإشارة إلى الضوضاء فعلًا، مع نسبة الكلام المستبقاة لإثبات عدم إتلاف المحتوى.

In [ ]:
pre = DIAG.set_index("fid")
after = []
for fid, grp in tqdm(QA.groupby("fid"), total=QA["fid"].nunique(),
                    desc="مقارنة قبل/بعد", unit="ملف"):
    # مقياس ثابت الجهارة: فجوة أرضية الضوضاء تحت مستوى الكلام (rms - floor)
    # مقيسة على الملف الكامل النهائي المصدَّر — عادلة عبر تغيّرات الجهارة
    fin = FULLS[FULLS["fid"] == fid]
    gap_after = np.nan
    if len(fin):
        fin_name = fin["full_name"].iloc[0]
        try:
            yf, srf = librosa.load(str(PATHS["out_full"] / fin_name),
                                   sr=None, mono=True)
            gap_after = round(rms_db(yf) - noise_floor_db(yf, srf), 1)
        except Exception as e:
            LOGGER.warning("تعذر قياس %s للمقارنة: %s", fin_name, e)
    gap_before = (pre.loc[fid, "floor_gap_db"]
                  if fid in pre.index and "floor_gap_db" in pre.columns else np.nan)
    after.append({
        "fid": fid,
        "snr_gap_before_db": round(gap_before, 1) if pd.notna(gap_before) else np.nan,
        "snr_gap_after_db": gap_after,
        "lufs_before": pre.loc[fid, "lufs"] if fid in pre.index else np.nan,
        "lufs_after": grp["lufs"].mean(),
        "clips": len(grp), "clips_s": grp["dur_s"].sum()})
COMPARE = pd.DataFrame(after)
if HAS_SILERO and "speech_pct" in DIAG.columns:
    COMPARE["speech_kept_pct"] = [
        round(min(100.0, r.clips_s / max(pre.loc[r.fid, "dur_s"]
              * (pre.loc[r.fid, "speech_pct"] / 100.0), 1e-9) * 100), 1)
        for r in COMPARE.itertuples()]
COMPARE["snr_gain_db"] = (COMPARE["snr_gap_after_db"]
                          - COMPARE["snr_gap_before_db"]).round(1)
COMPARE.to_csv(PATHS["out_qa"] / "before_after.csv",
               index=False, encoding="utf-8-sig")

n_pass = int(QA["pass_all"].sum())
print(f"النتيجة: {n_pass}/{len(QA)} مقطعًا يجتاز كل الفحوص "
      f"({100 * n_pass / max(len(QA), 1):.0f}%)")
if n_pass < len(QA):
    fails = QA[~QA["pass_all"]]
    print("مقاطع متعثرة (أول 10):")
    for _, r in fails.head(10).iterrows():
        bad = [c for c in CHECKS if not r[c]]
        print(f"  {r.clip_name}: {', '.join(bad)}")
print("\n== المقارنة قبل/بعد ==")
print(COMPARE.to_string(index=False))

### 12.1 المقارنة البصرية قبل/بعد

فحص بصري لعينة (أطول ملف): الموجة الخام في الأعلى مقابل الملف النهائي المنظف أدناه —
أين كانت الضوضاء/الصمت وأين أزيلت. تُحفظ نسخة PNG في `qa/`، وتُتخطى الخلية بأمان إن
تعذر التحميل.

In [ ]:
if HAS_MPL and len(QA):
    try:
        sample_fid = QA.sort_values("dur_s", ascending=False)["fid"].iloc[0]
        raw_row = REGISTRY[REGISTRY["fid"] == sample_fid].iloc[0]
        fin_name = FULLS[FULLS["fid"] == sample_fid]["full_name"].iloc[0]
        y_raw, sr_raw = librosa.load(raw_row.wav_path, sr=None, mono=True)
        y_fin, sr_fin = librosa.load(str(PATHS["out_full"] / fin_name),
                                     sr=None, mono=True)
        fig, axes = plt.subplots(2, 1, figsize=(13, 6), constrained_layout=True)
        t_raw = np.arange(len(y_raw)) / sr_raw
        t_fin = np.arange(len(y_fin)) / sr_fin
        axes[0].plot(t_raw, y_raw, lw=0.3, color="#888888")
        axes[0].set_title(f"BEFORE — raw: {raw_row.orig_name[:40]}")
        axes[1].plot(t_fin, y_fin, lw=0.3, color="#4C72B0")
        axes[1].set_title(f"AFTER — cleaned & normalized "
                          f"({CONFIG['target_lufs']} LUFS, {TARGET_SR} Hz)")
        for ax in axes:
            ax.set_xlabel("time (s)")
            ax.set_ylabel("amplitude")
        fig.savefig(PATHS["out_qa"] / "waveform_before_after.png", dpi=120)
        plt.show()
        print("عينة المقارنة:", raw_row.orig_name)
    except Exception as e:
        print("تخطي الرسم:", e)
elif not len(QA):
    print("لا مقاطع للعرض")

### 12.2 توزيعات المقاطع النهائية

توزيع مدد المقاطع (يلزم تركّزه في 3–12 ث) وجهارتها (يلزم تطابقها حول الهدف) — الفحص
البصري الأخير قبل التسليم.

In [ ]:
if HAS_MPL and len(QA):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    axes[0].hist(QA["dur_s"], bins=25, color="#4C72B0",
                 range=(0, CONFIG["max_clip_s"] + 1))
    axes[0].axvline(CONFIG["min_clip_s"], ls="--", c="k", label="min")
    axes[0].axvline(CONFIG["max_clip_s"], ls="--", c="r", label="max")
    axes[0].set_title("Final clip durations (s)")
    axes[0].legend()
    axes[1].hist(QA["lufs"], bins=25, color="#55A868")
    axes[1].axvline(CONFIG["target_lufs"], ls="--", c="k", label="target LUFS")
    axes[1].set_title("Final clip loudness (LUFS)")
    axes[1].legend()
    fig.savefig(PATHS["out_qa"] / "final_distributions.png", dpi=120)
    plt.show()

## 13 — المرحلة 11: بيان البيان (Manifest) والتغليف

**قابلية إعادة الإنتاج كاملة:** بيان JSON يجمد كل شيء — الإعدادات الفعلية، بصمة SHA-256
للأرشيف المصدر، إصدارات المكتبات، قرارات كل مرحلة (ما استُبعود ولماذا، نتائج الفحص
الاستباقي، المحرك المستخدم)، وسلسلة المنشأ لكل ملف نهائي — ثم حزمة ZIP واحدة جاهزة
للنقل إلى آلة التدريب.

In [ ]:
def file_meta(p):
    """بصمة وحجم ملف مخرج (مع فحص وجود صريح)."""
    p = Path(p)
    if not p.exists():
        return None
    try:
        return {"size_mb": round(p.stat().st_size / 1e6, 2),
                "sha256": sha256_of_file(p)}
    except OSError as e:
        LOGGER.warning("تعذر حساب بصمة %s: %s", p, e)
        return None

provenance = []
for fid in tqdm(KEPT["fid"], total=len(KEPT), desc="البيان", unit="ملف"):
    provenance.append({
        "fid": fid,
        "orig_name": REGISTRY[REGISTRY.fid == fid]["orig_name"].iloc[0],
        "stages": {
            "separated": str(VOC_PATHS.get(fid, "")).startswith(str(SEP_DIR)),
            "denoise_engine": (DEN_REPORT[DEN_REPORT.fid == fid]["engine"].iloc[0]
                               if len(DEN_REPORT) else "-"),
            "dereverb": (DRV_REPORT[DRV_REPORT.fid == fid]["status"].iloc[0]
                         if len(DRV_REPORT) else "-")},
        "clips": sorted(QA[QA.fid == fid]["clip_name"].tolist())})

MANIFEST = {
    "run_stamp": RUN_STAMP, "seed": CONFIG["seed"],
    "target": {"sr": TARGET_SR, "channels": 1, "subtype": "PCM_16",
               "lufs": CONFIG["target_lufs"]},
    "config_used": {k: v for k, v in CONFIG.items() if k != "zip_url"},
    "zip_sha256": sha256_of_file(ZIP_PATH),
    "env": {"python": ENV.get("python"), "gpu": ENV.get("gpu")},
    "library_versions": {"numpy": np.__version__, "librosa": librosa.__version__,
                         "soundfile": sf.__version__, "soxr": soxr.__version__},
    "pipeline_stats": {
        "files_raw": int(len(REGISTRY)), "files_kept": int(len(KEPT)),
        "files_dropped": dict(REASONS), "music_probe": MUSIC_FLAGS,
        "total_raw_min": round(float(REGISTRY["dur_s"].sum()) / 60, 1),
        "total_clean_min": round(float(QA["dur_s"].sum()) / 60, 1),
        "n_clips": int(len(QA)), "qa_pass": int(QA["pass_all"].sum())},
    "provenance": provenance,
    "outputs": {
        "clips": {p.name: file_meta(p)
                  for p in sorted(PATHS["out_clips"].glob("*.wav"))},
        "full_clean": {p.name: file_meta(p)
                       for p in sorted(PATHS["out_full"].glob("*.wav"))}}}

MANIFEST_PATH = PATHS["out"] / "manifest.json"
tmp_m = MANIFEST_PATH.with_suffix(".json.tmp")
tmp_m.write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=2, default=str),
                 encoding="utf-8")
tmp_m.replace(MANIFEST_PATH)

QA[["clip_name", "fid", "dur_s", "lufs", "peak_db", "pass_all"]].to_csv(
    PATHS["out"] / "dataset_index.csv", index=False, encoding="utf-8-sig")

if CONFIG["package_zip"]:
    zip_path = shutil.make_archive(str(PATHS["out"].parent / "rvc_dataset"),
                                    "zip", root_dir=str(PATHS["out"]))
    print("حزمة ZIP:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")

print("البيان:", MANIFEST_PATH)
print("\nشجرة المخرجات النهائية:")
print("\n".join(f"  {p.relative_to(PATHS['out'])}"
                for p in sorted(PATHS["out"].rglob("*")) if p.is_file()))

## 14 — الخلاصة وكيفية الاستخدام مع RVC v2

**ما أنتجته هذه الخطوط (في `output/rvc_dataset/`):**

- `clips/` — مقاطع نظيفة 3–12 ث، أحادية 48kHz/16-bit، جهارة موحدة، جاهزة للتدريب مباشرة.
- `full_clean/` — الملفات الكاملة بنفس المعايير (لو فضّلت تقطيع واجهة RVC نفسها).
- `qa/` — تقارير الفحص والمقارنات قبل/بعد | `manifest.json` + `dataset_index.csv` —
  البيان الكامل وفهرس المجموعة.

**خطوات التدريب:** في واجهة RVC/Applio: اختر مجلد `clips` كمجلد البيانات، معدل العينة
**48000**، النسخة **v2**، ثم مرّر التدريب المسبق (pretrained) المناسب لـ48k. مع ~30–40
دقيقة صوت نظيف تتوقع نتائج ممتازة؛ أضف تسجيلات جديدة للصوت نفسه إن أردت مزيدًا من
التماسك.

**نقاط صدق يجب معرفتها:**

- القص الرقمي (clipping) في التسجيل الأصلي **لا يُصلَح** بخط المعالجة — الملفات
  المتأثرة مُعلَّمة في تقارير التشخيص؛ استبدلها بتسجيلات سليمة إن توفرت.
- قرارات «هل فيه موسيقى؟» مبنية على قياس فعلي (فحص استباقي بنموذج فصل حقيقي)،
  ويمكن تجاوزها عبر `separation_mode` بقائمة معرفات.
- جرّب سماع 3–5 مقاطع عشوائية من `clips/` قبل التدريب — أذنك هي فحص الجودة الأخير.
- إعادة تشغيل الدفتر آمنة: المراحل المكتملة تُتخطى من الذاكرة المؤقتة؛ غيّر ما تشاء في
  `CONFIG` وأعد التشغيل.